# Deploy Jetbrains AI Mellum Model Package from AWS Marketplace


Mellum is JetBrains' first large language model (LLM) optimized for code-related tasks.

Designed for integration into professional developer tooling (e.g., intelligent code suggestions in IDEs), AI-powered coding assistants, and research on code understanding and generation.

This sample notebook shows you how to deploy any of the models from Mellum's family using Amazon SageMaker. For the sake of simplicity, we'll focus on [JetBrains AI Mellum All](https://aws.amazon.com/marketplace/pp/prodview-hwt2eytah3npu) in this notebook.

> **Note**: This is a reference notebook and it cannot run unless you make changes suggested in the notebook.

## Pre-requisites:
1. **Note**: This notebook contains elements which render correctly in Jupyter interface. Open this notebook from an Amazon SageMaker Notebook Instance or Amazon SageMaker Studio.
1. Ensure that IAM role used has **AmazonSageMakerFullAccess**


## Contents:
1. [Subscribe to the model package](#1.-Subscribe-to-the-model-package)
2. [Create an endpoint and perform real-time inference](#2.-Create-an-endpoint-and-perform-real-time-inference)
   1. [Create an endpoint](#A.-Create-an-endpoint)
   2. [Create input payload](#B.-Create-input-payload)
   3. [Perform real-time inference](#C.-Perform-real-time-inference)
   4. [Visualize output](#D.-Visualize-output)
   5. [Delete the endpoint](#E.-Delete-the-endpoint)
3. [Clean-up](#3.-Clean-up)
    1. [Delete the model](#A.-Delete-the-model)
    2. [Unsubscribe to the listing (optional)](#B.-Unsubscribe-to-the-listing-(optional))
    

## Usage instructions
You can run this notebook one cell at a time (By using Shift+Enter for running a cell).

## 1. Subscribe to the model package

To subscribe to the model package:
1. Open the model package listing page: [JetBrains AI Mellum All](https://aws.amazon.com/marketplace/pp/prodview-hwt2eytah3npu).
1. On the AWS Marketplace listing, click on the **Continue to subscribe** button.
1. On the **Subscribe to this software** page, review and click on **"Accept Offer"** if you and your organization agrees with EULA, pricing, and support terms. 
1. Once you click on **Continue to configuration button** and then choose a **region**, you will see a **Product Arn** displayed. This is the model package ARN that you need to specify while creating a deployable model using Boto3. Copy the ARN corresponding to your region and specify the same in the following cell.

In [ ]:
model_package_arn = "<Customer to specify Model package ARN corresponding to their AWS region>"

In [ ]:
import json

import sagemaker as sage
from sagemaker import get_execution_role
from sagemaker import ModelPackage
import boto3

In [ ]:
role = get_execution_role()

sagemaker_session = sage.Session()

bucket = sagemaker_session.default_bucket()
runtime = boto3.client("runtime.sagemaker")
bucket

## 2. Create an endpoint and perform real-time inference

If you want to understand how real-time inference with Amazon SageMaker works, see [Documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/how-it-works-hosting.html).

In [ ]:
model_name = "jbai-mellum-all"

content_type = "application/json"

# The recommended instance type for real-time inference is 'ml.g6e.xlarge'
# but it's sometimes challenging to acquire, so we use easier to get 'ml.g5.2xlarge'
real_time_inference_instance_type = "ml.g5.2xlarge"

#### A. Create an endpoint

In [ ]:
# create a deployable model from the model package.
model = ModelPackage(
    role=role, model_package_arn=model_package_arn, sagemaker_session=sagemaker_session
)

# Deploy the model
mellum_all = model.deploy(1, real_time_inference_instance_type, endpoint_name=model_name)

Once endpoint has been created, you would be able to perform real-time inference.

### B. Create input payload

In [ ]:
payload = {
  "prefix": "def main():\n    print(\"Hello {",
  "suffix": "",
  "filepath": "main.py",
  "context": [
      {
          "type": "DirectoryFile",
          "filepath": "settings.py",
          "content": "USER = 'cat'\n"
      }
  ],
  "max_length": 32,
  "stop_token": "\n\n",
  "use_control": "off"
}

### C. Perform real-time inference

In [ ]:
def run_inference(request, endpoint):
    raw_response = runtime.invoke_endpoint(
        EndpointName=endpoint,
        Body=json.dumps(request),
        ContentType="application/json",
    )
    status_code = raw_response["ResponseMetadata"]["HTTPStatusCode"]
    body = raw_response["Body"].read().decode()
    assert 200 <= status_code < 300, f"Request failed with the following message:\n {body}"

    messages = []
    for line in body.splitlines():
        if line.startswith("data:"):
            event_body = line.removeprefix("data: ")
            if event_body != "end":
                messages.append(json.loads(event_body))
    return messages

output = run_inference(payload, model_name)

### D. Visualize output

In [ ]:
print(output)

### E. Delete the endpoint

Now that you have successfully performed a real-time inference, you do not need the endpoint any more. You can terminate the endpoint to avoid being charged.

In [ ]:
model.sagemaker_session.delete_endpoint(model_name)
model.sagemaker_session.delete_endpoint_config(model_name)

## 3. Clean-up

### A. Delete the model

In [ ]:
model.delete_model()

### B. Unsubscribe to the listing (optional)

If you would like to unsubscribe to the model package, follow these steps. Before you cancel the subscription, ensure that you do not have any [deployable model](https://console.aws.amazon.com/sagemaker/home#/models) created from the model package or using the algorithm. Note - You can find this information by looking at the container name associated with the model. 

**Steps to unsubscribe to product from AWS Marketplace**:
1. Navigate to __Machine Learning__ tab on [__Your Software subscriptions page__](https://aws.amazon.com/marketplace/ai/library?productType=ml&ref_=mlmp_gitdemo_indust)
2. Locate the listing that you want to cancel the subscription for, and then choose __Cancel Subscription__  to cancel the subscription.

